Step 1: Bootstrap & install instructions

In [ ]:
!pip install --quiet anthropic pydantic
!pip install --upgrade --quiet ipython
!pip install --quiet langgraph
!pip install --quiet langchain_core

from google.colab import userdata, drive
drive.mount('/content/drive', force_remount=True)

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")
print('Ready!')


import importlib.metadata
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 1. Check version
try:
    print(f"✓ langgraph {importlib.metadata.version('langgraph')}")
except importlib.metadata.PackageNotFoundError:
    raise RuntimeError("langgraph not installed — run !pip install langgraph")

# 2. Build and run a minimal graph to verify imports and execution
class _State(TypedDict):
    status: str

def _node(state: _State) -> _State:
    return {"status": "ok"}

_g = StateGraph(_State)
_g.add_node("test", _node)
_g.add_edge(START, "test")
_g.add_edge("test", END)

result = _g.compile().invoke({"status": "start"})

if result.get("status") != "ok":
    raise AssertionError(f"Expected status 'ok', got {result}")

print("✓ Graph compiled and executed successfully!")

Step 2: Set up env vars and then import

In [ ]:
import os
from google.colab import userdata, drive


# Clear any lingering LangChain-prefixed vars from earlier attempts
for old in ("LANGCHAIN_TRACING_V2", "LANGCHAIN_API_KEY", "LANGCHAIN_PROJECT", "LANGCHAIN_ENDPOINT"):
    os.environ.pop(old, None)

# Set only the LangSmith-prefixed vars
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY").strip()
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["LANGSMITH_PROJECT"] = "astra-swarm-week5"

from astra_swarm import graph

# LangSmith Service Keys start with lsv2_sk_ or ls__ (varies by generation) and are 40–70 characters. If prefix is lsv2_pt_, that's a Personal Access Token, not a Service Key

import os
key = os.environ.get("LANGSMITH_API_KEY", "")
print(f"Key set: {bool(key)}")
print(f"Key length: {len(key)}")
print(f"Key prefix: {key[:8] if key else 'EMPTY'}...")
print(f"Key suffix: ...{key[-4:] if key else 'EMPTY'}")

Step 3: Run only once to create golden set

In [ ]:
from astra_swarm.cassette import cassette
from astra_swarm.alerts import _ask_structured
from pydantic import BaseModel, Field
from typing import Literal
from pathlib import Path
import json

class GoldenLabel(BaseModel):
    alert: str
    expected_routing: Literal["identity_auth", "malware", "network", "phishing", "other"]
    expected_severity: Literal["low", "medium", "high", "critical"]
    expected_escalation: bool
    expected_workers_contains: list[str] = Field(
        description="Workers that MUST run (e.g. ['enrichment_worker', 'itdr_specialist'])"
    )
    label_notes: str = Field(description="Why these labels")


class GoldenSet(BaseModel):
    labels: list[GoldenLabel]


# Load Week 1 alerts + a few extras if needed
alerts = json.loads(
    Path("/content/astra-swarm/data/synthetic/20_alerts.json").read_text()
)
# Generate 20 more diverse alerts if you only have 10
# (Reuse the Day 1 alert-generation pattern from Week 1)

with cassette("week5_golden_generation"):
    prompt = f"""You are labeling {len(alerts)} security alerts for an eval harness.
For each alert, produce:
- expected_routing: which specialist category the alert belongs to
- expected_severity: what a senior SOC analyst would set severity to
- expected_escalation: true if this warrants immediate escalation to a human on-call
- expected_workers_contains: which workers MUST run (enrichment_worker always;
  itdr_specialist if identity-involving; soc_analyst_worker if suspected campaign)
- label_notes: one sentence on the labeling reasoning

Alerts:
""" + "\n---\n".join(f"[{i}] {a}" for i, a in enumerate(alerts))

    labels = _ask_structured(prompt, GoldenSet, max_tokens=8000)

out = Path("/content/astra-swarm/evals/week5_golden_set.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps([l.model_dump() for l in labels.labels], indent=2))

Step 4: Create a dataset

In [ ]:
from langsmith import Client
import json
from pathlib import Path

client = Client()  # reads LANGCHAIN_API_KEY from env

DATASET_NAME = "astra-swarm-golden-v2"

# Delete existing dataset if you're iterating
try:
    existing = client.read_dataset(dataset_name=DATASET_NAME)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset {existing.id}")
except Exception:
    pass

# Load and upload
labels = json.loads(
    Path("/content/astra-swarm/evals/week5_golden_set.json").read_text()
)

dataset = client.create_dataset(
    dataset_name=DATASET_NAME,
    description="20-alert Astra-Swarm golden set with expected routing/severity/escalation/trajectory",
)

for label in labels:
    client.create_example(
        inputs={"alert": label["alert"]},
        outputs={
            "expected_routing": label["expected_routing"],
            "expected_severity": label["expected_severity"],
            "expected_escalation": label["expected_escalation"],
            "expected_workers_contains": label["expected_workers_contains"],
        },
        dataset_id=dataset.id,
    )

print(f"Uploaded {len(labels)} examples to dataset {dataset.id}")

Step 5: Evaluation

In [ ]:
from langsmith.evaluation import evaluate
from astra_swarm.graph import graph_triage
from astra_swarm.cassette import cassette
from astra_swarm.evaluators import (
    routing_correctness, severity_correctness,
    trajectory_correctness, escalation_correctness,
)


# Fixed the crash when result.get("investigation") returns None (guardrail-flagged path, or worker degradation) — because .model_dump() gets called on None.
def target(inputs: dict) -> dict:
    try:
        result = graph_triage(inputs["alert"])
    except Exception as e:
        print(f"Error: {e}")
        # Log the crash but return a scoreable shape so this example still gets scored (as zeros)
        return {
            "routing": None,
            "investigation": None,
            "workers_run": [],
            "escalated": False,
            "target_error": f"{type(e).__name__}: {str(e)[:200]}",
        }
    routing = result.get("routing")
    inv = result.get("investigation")
    return {
        "routing": routing.model_dump() if routing else None,
        "investigation": inv.model_dump() if inv else None,
        "workers_run": result.get("workers_run", []),
        "escalated": result.get("escalated", False),
        "guardrail_flagged": result.get("guardrail_flagged", False),
    }

with cassette("week5_eval_v2"):
    eval_result = evaluate(
        target,
        data=DATASET_NAME,
        evaluators=[
            routing_correctness,
            severity_correctness,
            trajectory_correctness,
            escalation_correctness,
        ],
        experiment_prefix="week5-baseline-v2",
        max_concurrency=1,   # be gentle on the API
    )

print(f"Eval done: {eval_result}")

Step 6: Diagnostics of eval report 

escalation_scores = []
routing_scores = []
severity_scores = []
trajectory_scores = []
per_example_data = []


for row in eval_result:  # iterable of result rows
    # Each row has feedback (evaluator scores) attached
    example = row["example"]
    run = row["run"]
    print("\n")
    print(example.inputs.get("alert"))
    # Find trajectory_correctness feedback for this row
    for feedback in row.get("evaluation_results", {}).get("results", []):
        if feedback.key == "routing_correctness":
            routing_scores.append(feedback.score)
            per_example_data.append({
                "expected_routing": example.outputs.get("expected_routing"),
                "alert": example.inputs.get("alert", "")[:150],
                "score": feedback.score,
                "comment": feedback.comment,
            })
        if feedback.key == "escalation_correctness":
            escalation_scores.append(feedback.score)
            per_example_data.append({
                "expected_routing": example.outputs.get("expected_routing"),
                "alert": example.inputs.get("alert", "")[:150],
                "score": feedback.score,
                "comment": feedback.comment,
            })
        if feedback.key == "severity_correctness":
            severity_scores.append(feedback.score)
            per_example_data.append({
                "expected_routing": example.outputs.get("expected_routing"),
                "alert": example.inputs.get("alert", "")[:150],
                "score": feedback.score,
                "comment": feedback.comment,
            })
        if feedback.key == "trajectory_correctness":
            trajectory_scores.append(feedback.score)
            per_example_data.append({
                "expected_routing": example.outputs.get("expected_routing"),
                "alert": example.inputs.get("alert", "")[:150],
                "score": feedback.score,
                "comment": feedback.comment,
            })

print("-" * 30)
print(f"Distribution ({len(trajectory_scores)} alerts) for routing_scores:")
print(f"  1.0 (perfect):     {sum(1 for s in routing_scores if s == 1.0)}")
print(f"  0.5 – 0.99:        {sum(1 for s in routing_scores if 0.5 <= s < 1.0)}")
print(f"  0.01 – 0.49:       {sum(1 for s in routing_scores if 0 < s < 0.5)}")
print(f"  0.0 (no overlap):  {sum(1 for s in routing_scores if s == 0.0)}")
print(f"Mean: {sum(routing_scores)/len(routing_scores):.3f}")
print(f"Median: {sorted(routing_scores)[len(routing_scores)//2]:.3f}")

print("-" * 30)
print(f"Distribution ({len(trajectory_scores)} alerts) for escalation_scores:")
print(f"  1.0 (perfect):     {sum(1 for s in escalation_scores if s == 1.0)}")
print(f"  0.5 – 0.99:        {sum(1 for s in escalation_scores if 0.5 <= s < 1.0)}")
print(f"  0.01 – 0.49:       {sum(1 for s in escalation_scores if 0 < s < 0.5)}")
print(f"  0.0 (no overlap):  {sum(1 for s in escalation_scores if s == 0.0)}")
print(f"Mean: {sum(escalation_scores)/len(escalation_scores):.3f}")
print(f"Median: {sorted(escalation_scores)[len(escalation_scores)//2]:.3f}")

print("-" * 30)
print(f"Distribution ({len(severity_scores)} alerts) for severity_scores:")
print(f"  1.0 (perfect):     {sum(1 for s in severity_scores if s == 1.0)}")
print(f"  0.5 – 0.99:        {sum(1 for s in severity_scores if 0.5 <= s < 1.0)}")
print(f"  0.01 – 0.49:       {sum(1 for s in severity_scores if 0 < s < 0.5)}")
print(f"  0.0 (no overlap):  {sum(1 for s in severity_scores if s == 0.0)}")
print(f"Mean: {sum(severity_scores)/len(severity_scores):.3f}")
print(f"Median: {sorted(severity_scores)[len(severity_scores)//2]:.3f}")

print("-" * 30)
print(f"Distribution ({len(trajectory_scores)} alerts) for trajectory_scores:")
print(f"  1.0 (perfect):     {sum(1 for s in trajectory_scores if s == 1.0)}")
print(f"  0.5 – 0.99:        {sum(1 for s in trajectory_scores if 0.5 <= s < 1.0)}")
print(f"  0.01 – 0.49:       {sum(1 for s in trajectory_scores if 0 < s < 0.5)}")
print(f"  0.0 (no overlap):  {sum(1 for s in trajectory_scores if s == 0.0)}")
print(f"Mean: {sum(trajectory_scores)/len(trajectory_scores):.3f}")
print(f"Median: {sorted(trajectory_scores)[len(trajectory_scores)//2]:.3f}")
print("-" * 50)

# Three lowest for case review
lowest = sorted(per_example_data, key=lambda x: x["score"])[:3]
for ex in lowest:
    print(f"\n[{ex['expected_routing']}] score={ex['score']:.2f}")
    print(f"    alert: {ex['alert']}...")
    print(f"    comment: {ex['comment']}")

print("Finished diagnostic of eval repo")

Step 7: Per example scores and overlap

In [ ]:
from langsmith import Client
client = Client()

# Fetch the experiment's example-level results
EXPERIMENT_NAME = eval_result.experiment_name  # e.g. "week5-baseline-abc123"

# Collect per-example scores by iterating the results
routing_fails = []
escalation_fails = []
per_example_scores = {}   # {example_id: {metric_key: score}}

for row in eval_result:
    example_id = str(row["example"].id)
    per_example_scores[example_id] = {
        "alert": row["example"].inputs.get("alert", "")[:200],
        "expected_routing": row["example"].outputs.get("expected_routing"),
        "expected_escalation": row["example"].outputs.get("expected_escalation"),
        "predicted_routing": (row["run"].outputs.get("routing") or {}).get("alert_class"),
        "predicted_escalated": row["run"].outputs.get("escalated"),
        "scores": {},
    }
    for fb in row.get("evaluation_results", {}).get("results", []):
        per_example_scores[example_id]["scores"][fb.key] = {
            "score": fb.score,
            "comment": fb.comment,
        }
        if fb.key == "routing_correctness" and fb.score == 0.0:
            routing_fails.append(example_id)
        if fb.key == "escalation_correctness" and fb.score == 0.0:
            escalation_fails.append(example_id)

overlap = set(routing_fails) & set(escalation_fails)
print(f"Routing fails: {len(routing_fails)} alerts")
print(f"Escalation fails: {len(escalation_fails)} alerts")
print(f"Overlap: {len(overlap)} alerts fail both")

print(f"\n=== Routing failures ({len(routing_fails)}) ===\n")
for eid in routing_fails:
    ex = per_example_scores[eid]
    print(f"Alert: {ex['alert']}...")
    print(f"  Expected routing:  {ex['expected_routing']}")
    print(f"  Predicted routing: {ex['predicted_routing']}")
    print(f"  Escalation: expected={ex['expected_escalation']} predicted={ex['predicted_escalated']}")
    print(f"  Routing comment: {ex['scores'].get('routing_correctness', {}).get('comment', '')}")
    print()

Step: Fast diagnostic for trajectory_correctness = 1

In [ ]:
import json
from pathlib import Path
from astra_swarm.graph import graph_triage

labels = json.loads(Path("/content/astra-swarm/evals/week5_golden_set.json").read_text())

for i, label in enumerate(labels[:5]):
    result = graph_triage(label["alert"])
    workers = {w.split(":")[0] for w in result.get("workers_run", [])}
    required = set(label["expected_workers_contains"])
    extra = workers - required
    missing = required - workers
    print(f"[{i+1}] routing={label['expected_routing']}")
    print(f"    required: {sorted(required)}")
    print(f"    ran:      {sorted(workers)}")
    print(f"    extra:    {sorted(extra)}   missing: {sorted(missing)}")
    print()